# Xenium data plotting
##UMAP, composition plots, barplots and spatial plotsAnalyses and plotting code used to generate Figure 2 and associated supplementaryvisualizations. Expects the processed object produced by `Xenium01` preprocessing.

## Setup

In [ ]:
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.font_manager import FontProperties
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# TrueType fonts in PDF/PS output (editable in Illustrator)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

In [ ]:
# ---------------------------------------------------------------------------
# Paths and output settings
# ---------------------------------------------------------------------------

IN_H5AD = Path(
    "../../../combined_Xenium_largecells_noWT3_afterclustering_withleiden_"
    "annotated_ordered_withTcells_ReannotatedIfgga4.h5ad"
)

OUT_H5AD = Path(
    "../../combined_Xenium_largecells_noWT3_afterclustering_withleiden_"
    "annotated_ordered_withTcells_ReannotatedIfgga4_newcolours_test.h5ad"
)

OUT_DIR = Path("figures_fig2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = OUT_DIR
sc.settings.set_figure_params(dpi_save=300, transparent=True)

# obs column names, used throughout
CELL_COL = "celltype3"
SAMPLE_COL = "name"
GROUP_COL = "type"

print("input :", IN_H5AD.resolve())
print("output:", OUT_DIR.resolve())

## Figure colours and category orderThe colour dictionary and the category order are the single source of truth forevery panel. `sync_colors` rewrites `.uns[f"{key}_colors"]` from the *current*categories, so it must be called again after any reordering or subsetting.

In [ ]:
CELL_STATE_COLORS = {
    "Basal VCMs":          "#2C7BB6",  # blue
    "IFN-associated VCMs": "#FFD92F",  # gold
    "Stressed VCMs":       "#D7191C",  # red
    "Remodelled VCMs":     "#1A9641",  # green
    "FBs":                 "#984EA3",  # purple
    "Vasculature ECs":     "#00A6CA",  # cyan
    "Endocardial ECs":     "#FDB863",  # light orange
    "LECs":                "#4DBBD5",  # light blue
    "Epicardium":          "#35978F",  # teal
    "Pericytes":           "#A6761D",  # dark gold
    "SMCs":                "#80CDC1",  # mint
    "Myeloid":             "#FF7A00",  # orange
    "T cells":             "#7B2CFF",  # violet
    "B cells":             "#FF4FA3",  # pink
    "Lymphoid":            "#5E3AB8",  # dark violet
    "NCs":                 "#8073AC",  # light violet
    "ACMs":                "#8C510A",  # brown
}

# Previously "Lymphoid" and "T cells" shared a colour and "B cells" was listed
# twice; both are fixed above. Adjust "Lymphoid" if you do not use that label.


def sync_colors(ad, key, cmap=CELL_STATE_COLORS, uns_key=None):
    """Write .uns colours matching the *current* category order of ad.obs[key].

    scanpy maps uns[f"{key}_colors"][i] onto categories[i] positionally, so this
    has to be re-run whenever categories are reordered, renamed or dropped.
    """
    cats = list(ad.obs[key].cat.categories)
    missing = [c for c in cats if c not in cmap]
    if missing:
        raise KeyError(f"no colour defined for: {missing}")
    ad.uns[uns_key or f"{key}_colors"] = [cmap[c] for c in cats]
    return ad.uns[uns_key or f"{key}_colors"]

## Load processed data

In [ ]:
combined_adata = sc.read_h5ad(IN_H5AD)
print(combined_adata)
print("celltype3:", list(combined_adata.obs["celltype3"].cat.categories))
print("celltype4:", list(combined_adata.obs["celltype4"].cat.categories))

In [ ]:
# ---------------------------------------------------------------------------
# Rename cell states to their final figure labels
# ---------------------------------------------------------------------------

RENAME = {
    "VCMs": "Basal VCMs",
    "Myh7+ VCMs": "Remodelled VCMs",
    "Ifgga4+ VCMs": "IFN-associated VCMs",
}

for col in ["celltype3", "celltype4"]:
    present = {k: v for k, v in RENAME.items()
               if k in combined_adata.obs[col].cat.categories}
    if present:
        combined_adata.obs[col] = combined_adata.obs[col].cat.rename_categories(present)

print("celltype3:", list(combined_adata.obs["celltype3"].cat.categories))

In [ ]:
# ---------------------------------------------------------------------------
# Category order
#
# pd.Categorical silently converts any value not listed in `categories` to NaN,
# which would drop those cells from every count and plot downstream. The
# assertions below turn that into a loud failure.
# ---------------------------------------------------------------------------

CELLTYPE_ORDER = [
    "Basal VCMs", "IFN-associated VCMs", "Stressed VCMs", "Remodelled VCMs",
    "FBs", "Vasculature ECs", "Endocardial ECs", "Epicardium",
    "Myeloid", "T cells", "B cells",
    "Pericytes", "SMCs", "NCs", "ACMs",
]

GROUP_ORDER = ["WT", "BE", "R636Q"]

SAMPLE_ORDER = [
    "WT_rep1", "WT_rep2", "WT_rep4", "WT_rep5",
    "BE_rep1", "BE_rep2", "BE_rep3", "BE_rep4",
    "PBS_rep1", "PBS_rep2", "PBS_rep3", "PBS_rep4",
]


def set_order(ad, col, order, ordered=True):
    observed = set(ad.obs[col].dropna().astype(str).unique())
    unlisted = observed - set(order)
    assert not unlisted, f"{col}: values present in the data but not in the order list: {sorted(unlisted)}"
    unused = set(order) - observed
    if unused:
        print(f"note: {col} order lists categories with no cells: {sorted(unused)}")
    ad.obs[col] = pd.Categorical(ad.obs[col].astype(str), categories=order, ordered=ordered)


set_order(combined_adata, CELL_COL, CELLTYPE_ORDER)
set_order(combined_adata, GROUP_COL, GROUP_ORDER)
set_order(combined_adata, SAMPLE_COL, SAMPLE_ORDER)

assert combined_adata.obs[[CELL_COL, GROUP_COL, SAMPLE_COL]].notna().all().all(), \
    "NaNs introduced while setting categorical order"

In [ ]:
# ---------------------------------------------------------------------------
# Colours LAST, once the category order is final
# ---------------------------------------------------------------------------

sync_colors(combined_adata, "celltype3")
sync_colors(combined_adata, "celltype4")

for ct, col in zip(combined_adata.obs[CELL_COL].cat.categories,
                   combined_adata.uns["celltype3_colors"]):
    print(f"{ct:<24} {col}")

In [ ]:
combined_adata.write_h5ad(OUT_H5AD)
print("written:", OUT_H5AD.resolve())

## Cell-type composition per sample and per group`group_means` averages the per-sample proportions rather than pooling cells

In [ ]:
obs_ct = combined_adata.obs[[CELL_COL, SAMPLE_COL, GROUP_COL]].copy()

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs_ct[SAMPLE_COL], obs_ct[CELL_COL], normalize="index", dropna=False)
    .reindex(index=SAMPLE_ORDER, columns=CELLTYPE_ORDER)
    .fillna(0)
)

# mean proportions per group
sample2group = (
    obs_ct.drop_duplicates(SAMPLE_COL)
    .set_index(SAMPLE_COL)[GROUP_COL]
    .reindex(SAMPLE_ORDER)
)
assert sample2group.notna().all(), "some samples have no group assignment"

group_means = (
    props_sample.assign(**{GROUP_COL: sample2group.values})
    .groupby(GROUP_COL, observed=True)[CELLTYPE_ORDER]
    .mean()
    .reindex(index=GROUP_ORDER)
)

group_means.to_csv(OUT_DIR / "group_means_Xenium_celltypes_Ifgga4reannotated.csv")
props_sample.to_csv(OUT_DIR / "samplenoWT3_Xenium_celltypes_Ifgga4reannotated.csv")

group_means.round(3)

In [ ]:
def stacked_barplot(df, filename, ylabel, celltype_order=CELLTYPE_ORDER,
                    cmap=CELL_STATE_COLORS, figsize=(8, 5), width=0.9):
    """Stacked composition barplot. Bars stack bottom-up in reversed order so the
    first cell type sits on top; the legend keeps the forward order."""
    stack_order = list(celltype_order)[::-1]

    fig, ax = plt.subplots(figsize=figsize)
    df[stack_order].plot(
        kind="bar", stacked=True, ax=ax,
        color=[cmap[ct] for ct in stack_order], width=width,
    )

    ax.legend(
        handles=[Patch(facecolor=cmap[ct], label=ct) for ct in celltype_order],
        title="celltype", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False,
    )
    ax.grid(False)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    plt.tight_layout()

    fig.savefig(OUT_DIR / filename, dpi=300, bbox_inches="tight", transparent=True)
    return fig, ax

In [ ]:
fig, ax = stacked_barplot(
    props_sample,
    "celltype_proportions_per_sample_Ifgga4reannotated.pdf",
    ylabel="Cell type proportion",
    figsize=(8, 5),
)
plt.show()

In [ ]:
fig, ax = stacked_barplot(
    group_means,
    "celltype_mean_proportions_per_group_Ifgga4reannotated.pdf",
    ylabel="Mean cell type proportion",
    figsize=(5, 5),
)
plt.show()

## UMAP

In [ ]:
sc.pl.umap(
    combined_adata,
    color=CELL_COL,
    palette=CELL_STATE_COLORS,
    save="_noWT3_withT_Bcells_reannotatedIfgga4.pdf",
)

## VCM subset: marker dotplots and subtype composition`sc.pl.dotplot` is used rather than `sc.pl.rank_genes_groups_dotplot`: the genesare specified explicitly, and the subset inherits an `uns["rank_genes_groups"]`computed on the full object with a different grouping.

In [ ]:
VCM_CELLTYPES = ["Basal VCMs", "IFN-associated VCMs", "Stressed VCMs", "Remodelled VCMs"]

VCMs = combined_adata[combined_adata.obs[CELL_COL].isin(VCM_CELLTYPES)].copy()
VCMs.obs[CELL_COL] = VCMs.obs[CELL_COL].cat.remove_unused_categories()
sync_colors(VCMs, CELL_COL)

print(VCMs.obs[CELL_COL].value_counts())

In [ ]:
# Defined once, with the current (renamed) labels. These keys become the gene
# group brackets above the dotplot columns.
genes_by_group = {
    "Basal VCMs": ["Ttn", "Myl2", "Tnnt2", "Ttn_N2B", "Ttn_flanking-exons", "Camk2d_isoformA"],
    "IFN-associated VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons", "Camk2d_isoformA"],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon", "Camk2d_isoformB"],
    "Remodelled VCMs": ["Myh7"],
}

missing_genes = sorted({g for gs in genes_by_group.values() for g in gs
                        if g not in VCMs.var_names})
assert not missing_genes, f"genes not in var_names: {missing_genes}"

In [ ]:
sc.pl.dotplot(
    VCMs,
    genes_by_group,
    groupby=CELL_COL,
    categories_order=VCM_CELLTYPES,
    standard_scale="var",
    dendrogram=False,
    save="mygenes_grouped_VCMsspatial_Ifgga4_reannotated.pdf",
)

sc.pl.dotplot(
    VCMs,
    genes_by_group,
    groupby=CELL_COL,
    categories_order=VCM_CELLTYPES,
    dendrogram=False,
    save="mygenes_grouped_VCMsspatial_Ifgga4_reannotated_novarscale.pdf",
)

In [ ]:
# Cardiomyocyte subtype proportions, rescaled so VCM subtypes sum to 1 per sample
cm_celltypes = [ct for ct in VCM_CELLTYPES if ct in props_sample.columns]

props_cm_sample = props_sample[cm_celltypes].copy()
props_cm_sample = props_cm_sample.div(props_cm_sample.sum(axis=1), axis=0).fillna(0)

fig, ax = stacked_barplot(
    props_cm_sample,
    "cardiomyocyte_subtype_proportions_per_sample.pdf",
    ylabel="Proportion of cardiomyocytes",
    celltype_order=cm_celltypes,
    figsize=(8, 5),
)
ax.legend_.set_title("Cardiomyocyte subtype")
plt.show()

## Long-format composition tablePercentages are of all cells in the section, so the focus cell types do not haveto sum to 100%.

In [ ]:
FOCUS_CELLTYPES = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "FBs",
    "Myeloid",
    "T cells",
]

GROUP_PALETTE = {
    "WT": "#2C7BB6",
    "BE": "#FFD92F",
    "R636Q": "#D7191C",
}

In [ ]:
obs_long = combined_adata.obs[[SAMPLE_COL, GROUP_COL, CELL_COL]].astype(str).copy()

counts = (
    obs_long.groupby([SAMPLE_COL, GROUP_COL, CELL_COL], observed=True)
    .size().reset_index(name="n_cells")
)

totals = (
    obs_long.groupby([SAMPLE_COL, GROUP_COL], observed=True)
    .size().reset_index(name="total_cells")
)

# every focus cell type for every real sample/group combination, zero-filled
celltype_df = pd.DataFrame({CELL_COL: FOCUS_CELLTYPES})
composition = (
    totals.merge(celltype_df, how="cross")
    .merge(counts, on=[SAMPLE_COL, GROUP_COL, CELL_COL], how="left")
)
composition["n_cells"] = composition["n_cells"].fillna(0)
composition["percent"] = composition["n_cells"] / composition["total_cells"] * 100

composition[SAMPLE_COL] = pd.Categorical(composition[SAMPLE_COL], SAMPLE_ORDER, ordered=True)
composition[GROUP_COL] = pd.Categorical(composition[GROUP_COL], GROUP_ORDER, ordered=True)
composition[CELL_COL] = pd.Categorical(composition[CELL_COL], FOCUS_CELLTYPES, ordered=True)

assert len(composition) == len(totals) * len(FOCUS_CELLTYPES)
composition.to_csv(OUT_DIR / "composition_long.csv", index=False)
composition.head()

In [ ]:
composition_wide = composition.pivot(
    index=SAMPLE_COL, columns=CELL_COL, values="percent"
).reindex(index=SAMPLE_ORDER)

composition_wide.round(2)

## StatisticsComputed once, over the full set of cell types being reported, so the BH-FDRfamily does not depend on which panel happens to be drawn. The plotting functionconsumes this table rather than recomputing it.Two caveats worth keeping in mind at n = 4 sections per group: `mannwhitneyu`two-sided has a floor of p ≈ 0.029, so after correction nothing will reachsignificance; and percentages are bounded, so a logit or arcsine-sqrt transform(or a compositional method such as scCODA / propeller) is the more defensibleroute if a reviewer pushes on it.

In [ ]:
def p_to_star(p):
    if p < 0.0001:
        return "****"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def calculate_pairwise_tests(data, celltypes, comparisons, test="ttest",
                             celltype_col=CELL_COL, condition_col=GROUP_COL,
                             value_col="percent"):
    """Pairwise group comparisons per cell type, with one BH-FDR family over all
    tests in the returned table.

    test: 'ttest' (Welch) or 'mannwhitney'.
    """
    rows = []
    for ct in celltypes:
        sub = data[data[celltype_col] == ct]
        for g1, g2 in comparisons:
            x = sub.loc[sub[condition_col] == g1, value_col].dropna()
            y = sub.loc[sub[condition_col] == g2, value_col].dropna()

            if len(x) < 2 or len(y) < 2:
                p = np.nan
            elif test == "ttest":
                _, p = ttest_ind(x, y, equal_var=False)
            elif test == "mannwhitney":
                _, p = mannwhitneyu(x, y, alternative="two-sided")
            else:
                raise ValueError("test must be 'ttest' or 'mannwhitney'")

            rows.append({
                celltype_col: ct, "comparison": f"{g1} vs {g2}",
                "group1": g1, "group2": g2,
                "n1": len(x), "n2": len(y),
                "mean1": x.mean() if len(x) else np.nan,
                "mean2": y.mean() if len(y) else np.nan,
                "pvalue": p,
            })

    results = pd.DataFrame(rows)
    results["padj"] = np.nan
    valid = results["pvalue"].notna()
    if valid.any():
        results.loc[valid, "padj"] = multipletests(
            results.loc[valid, "pvalue"], method="fdr_bh"
        )[1]
    results["stars"] = results["padj"].apply(lambda p: p_to_star(p) if pd.notna(p) else "")
    return results

In [ ]:
COMPARISONS = [("WT", "R636Q"), ("R636Q", "BE"), ("WT", "BE")]
TEST = "ttest"   # or "mannwhitney"

stats_all = calculate_pairwise_tests(
    composition, celltypes=FOCUS_CELLTYPES, comparisons=COMPARISONS, test=TEST
)
stats_all.to_csv(OUT_DIR / f"composition_stats_{TEST}_bh.csv", index=False)
stats_all.round(4)

## Composition plots

In [ ]:
def plot_composition(data, celltypes, stats=None, title="Cell-state composition",
                     ylimit=None, figsize=(8, 5), size_scale=1.0,
                     celltype_col=CELL_COL, condition_col=GROUP_COL,
                     condition_order=GROUP_ORDER, palette=GROUP_PALETTE,
                     rng_seed=0):
    """Per-section points with mean +/- SEM, plus significance brackets taken from
    a precomputed `stats` table (output of calculate_pairwise_tests)."""
    rng = np.random.default_rng(rng_seed)
    plot_data = data[data[celltype_col].isin(celltypes)].copy()

    fig, ax = plt.subplots(figsize=figsize)
    x_positions = np.arange(len(celltypes))
    width = 0.22
    offsets = dict(zip(condition_order, [-width, 0.0, width]))

    for condition in condition_order:
        cond_data = plot_data[plot_data[condition_col] == condition]
        for i, ct in enumerate(celltypes):
            vals = cond_data.loc[cond_data[celltype_col] == ct, "percent"].to_numpy()
            x = x_positions[i] + offsets[condition]

            ax.scatter(
                np.repeat(x, len(vals)) + rng.normal(0, 0.025, size=len(vals)),
                vals,
                color=palette[condition], s=30 * size_scale, alpha=0.9,
                label=condition if i == 0 else None,
            )

            if len(vals):
                sem = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
                ax.errorbar(
                    x, np.mean(vals), yerr=sem, color=palette[condition],
                    fmt="_", markersize=18, capsize=4 * size_scale,
                    capthick=2 * size_scale, elinewidth=2 * size_scale,
                )

    # ---- significance brackets --------------------------------------------
    y_top = ylimit[1] if ylimit is not None else plot_data["percent"].max() * 1.2
    y_step = y_top * 0.07
    highest = -np.inf

    if stats is not None:
        for i, ct in enumerate(celltypes):
            sig = stats[(stats[celltype_col] == ct) & (~stats["stars"].isin(["ns", ""]))]
            ct_vals = plot_data.loc[plot_data[celltype_col] == ct, "percent"]
            base_y = (ct_vals.max() if len(ct_vals) else 0) + y_step

            # `level` counts only the brackets actually drawn, so skipping a
            # non-significant comparison does not leave a floating gap
            for level, (_, row) in enumerate(sig.iterrows()):
                y = base_y + level * y_step
                x1 = x_positions[i] + offsets[row["group1"]]
                x2 = x_positions[i] + offsets[row["group2"]]

                ax.plot(
                    [x1, x1, x2, x2],
                    [y, y + y_step * 0.2, y + y_step * 0.2, y],
                    color="black", linewidth=1 * size_scale,
                )
                ax.text(
                    (x1 + x2) / 2, y + y_step * 0.25, row["stars"],
                    ha="center", va="bottom", fontsize=10 * size_scale,
                )
                highest = max(highest, y + y_step * 0.6)

    ax.set_title(title, fontsize=18)
    ax.set_ylabel("Cells per section (%)", fontsize=14)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(celltypes, rotation=40, ha="right", fontsize=12)
    ax.legend(frameon=False, fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    if ylimit is not None:
        # expand rather than clip if a bracket sits above the requested limit
        ax.set_ylim(ylimit[0], max(ylimit[1], highest))
    plt.tight_layout()
    return fig, ax

In [ ]:
main_celltypes = ["Basal VCMs", "IFN-associated VCMs", "Stressed VCMs", "FBs"]

fig, ax = plot_composition(
    data=composition,
    celltypes=main_celltypes,
    stats=stats_all,
    title="Cell-state composition",
    ylimit=(-1, 70),
    figsize=(6, 7),
    size_scale=0.65,
)
fig.savefig(OUT_DIR / "cell_state_composition.pdf", transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
immune_celltypes = ["T cells", "Myeloid"]

fig, ax = plot_composition(
    data=composition,
    celltypes=immune_celltypes,
    stats=stats_all,
    title="Cell-state composition",
    ylimit=(-1, 10),
    figsize=(4, 7),
    size_scale=0.65,
)
fig.savefig(OUT_DIR / "cell_state_composition_Tcells_myeloid.pdf",
            transparent=True, bbox_inches="tight")
plt.show()

## Spatial plots Two panels
BE sections with the VCM states shown and IFN-associated VCMs drawnon top of everything else, and one figure per sample with the IFN / T cell /myeloid populations brought to the front.`sync_colors` is called on every subset. Subsetting an AnnData drops unusedcategories, which leaves `uns[..._colors]` the wrong length; scanpy then fallsback to a default palette, and colours would differ between panels.

In [ ]:
adata = sc.read_h5ad(OUT_H5AD)
print(adata)

In [ ]:
def bring_to_front(ad, cell_col, front_cts, highlight_ct=None):
    """Return a view reordered so front_cts are drawn last (and highlight_ct
    last of all). Stable sort keeps within-layer order reproducible."""
    ct = ad.obs[cell_col].astype(str)
    rank = np.select(
        [ct.eq(highlight_ct) if highlight_ct else np.zeros(len(ct), bool),
         ct.isin(front_cts)],
        [2, 1],
        default=0,
    )
    return ad[np.argsort(rank, kind="stable")].copy()


def add_scalebar(ax, scalebar_um=1000, size_vertical=10, fontsize=12, **kwargs):
    ax.add_artist(AnchoredSizeBar(
        ax.transData, scalebar_um, f"{scalebar_um} \u00b5m", loc="lower right",
        pad=0.4, borderpad=0.5, sep=4, color="black", frameon=False,
        size_vertical=size_vertical, fontproperties=FontProperties(size=fontsize),
        **kwargs,
    ))


def clean_spatial_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_frame_on(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

In [ ]:
# ---------------------------------------------------------------------------
# BE panel: VCM states only, IFN-associated VCMs highlighted
# ---------------------------------------------------------------------------

selected_cts = ["Basal VCMs", "IFN-associated VCMs", "Stressed VCMs", "Remodelled VCMs"]
highlight_ct = "IFN-associated VCMs"
highlight_color = CELL_STATE_COLORS[highlight_ct]

be_samples = [s for s in SAMPLE_ORDER if s in set(adata.obs.loc[adata.obs[GROUP_COL] == "BE", SAMPLE_COL])]

fig, axes = plt.subplots(1, len(be_samples), figsize=(5 * len(be_samples), 5))
axes = np.atleast_1d(axes)

for ax, sample in zip(axes, be_samples):
    sub = adata[adata.obs[SAMPLE_COL] == sample].copy()

    # set_categories drops everything else to NaN -> plotted as na_color
    sub.obs[CELL_COL] = sub.obs[CELL_COL].cat.set_categories(selected_cts)
    sync_colors(sub, CELL_COL)

    sub = bring_to_front(sub, CELL_COL, selected_cts, highlight_ct)
    sync_colors(sub, CELL_COL)

    sc.pl.spatial(
        sub, color=CELL_COL, na_color="lightgrey", na_in_legend=False,
        spot_size=25, show=False, ax=ax,
    )

    # redraw the highlighted cells on top so they stay visible
    mask = sub.obs[CELL_COL].astype(str).eq(highlight_ct).to_numpy()
    coords = sub.obsm["spatial"][mask]
    ax.scatter(coords[:, 0], coords[:, 1], s=3, c=highlight_color, edgecolors="none")

    ax.set_title(sample)
    clean_spatial_axes(ax)
    add_scalebar(ax, scalebar_um=1000, size_vertical=10, fontsize=10)

plt.tight_layout()
fig.savefig(OUT_DIR / "BE_all_samples_VCMs_highlighted_panel.pdf",
            bbox_inches="tight", dpi=600)
plt.show()
plt.close(fig)

In [ ]:
# ---------------------------------------------------------------------------
# One figure per sample, identical axis limits everywhere
#
# The previous version used ylim (5500, -500) for the BE sections and
# (6000, -500) for WT/PBS despite the comment saying the values were identical,
# so the fields of view were not comparable. One value is used for all samples
# below; override per sample in YLIM_OVERRIDES only if a section is clipped.
# ---------------------------------------------------------------------------

TARGET_CTS = ["IFN-associated VCMs", "T cells", "Myeloid"]

COMMON_XLIM = (-500, 5500)
COMMON_YLIM = (6000, -500)   # inverted y: image convention
YLIM_OVERRIDES = {}

FIGURE_SIZE = (8, 8)
SPOT_SIZE = 25


def plot_sample_spatial(ad, sample, target_cts=TARGET_CTS, suffix="targets_on_top_IfggaTMy_scalebar",
                        xlim=COMMON_XLIM, ylim=None, figsize=FIGURE_SIZE,
                        spot_size=SPOT_SIZE, dpi=600, save=True):
    ylim = ylim or YLIM_OVERRIDES.get(sample, COMMON_YLIM)

    sub = ad[ad.obs[SAMPLE_COL] == sample].copy()
    sub.obs[CELL_COL] = sub.obs[CELL_COL].cat.remove_unused_categories()
    sub = bring_to_front(sub, CELL_COL, target_cts)
    sync_colors(sub, CELL_COL)

    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.spatial(sub, color=CELL_COL, na_color="lightgrey",
                  spot_size=spot_size, ax=ax, show=False)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(sample)

    add_scalebar(ax, scalebar_um=1000, size_vertical=10, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

    # identical margins across samples
    fig.subplots_adjust(left=0.05, right=0.82, bottom=0.05, top=0.92)

    if save:
        fig.savefig(OUT_DIR / f"{sample}_{suffix}.pdf", dpi=dpi)
    return fig, ax

In [ ]:
for sample in SAMPLE_ORDER:
    fig, ax = plot_sample_spatial(adata, sample)
    plt.show()
    plt.close(fig)   # 12 open figures otherwise